# `cluster_topics` — User Guide

Daily topic clustering pipeline that discovers emergent market narratives from RSS feed articles and tracks their frequency as a trading signal.

```
cluster_topics.py
│
├── Core clustering (no API key)
│   ├── reduce_dimensions(vectors, n_components=50)   → np.ndarray
│   └── run_hdbscan(X, ...)                           → (labels, noise_ratio) | ClusteringAborted
│
├── Topic continuity (no API key)
│   ├── compute_centroids(vectors, labels)            → dict[int, ndarray]
│   ├── match_topics(new_centroids, prior, ...)       → dict[int, dict]
│   ├── load_centroids(path) / save_centroids(...)    → dict / None
│   └── load_label_cache(path) / save_label_cache(...) → dict / None
│
├── LLM labeling (requires OPENAI_API_KEY)
│   └── get_label(topic_id, cache, articles, llm_fn)  → str
│
├── Time-series output (no API key)
│   └── append_trends(run_date, rows, path)           → None | DuplicateDateError
│
├── Signal generation (no API key)
│   ├── compute_spike(topic_id, trends, target_date, ...) → float | None
│   └── get_emerging_topics(target_date, trends, ...)     → list[dict]
│
└── Full pipeline (requires OPENAI_API_KEY + FAISS store)
    └── run(target_date, ...)                         → summary dict
```

| Function | Needs FAISS | Needs LLM | Best for |
|---|---|---|---|
| `reduce_dimensions` | no | no | standalone dim reduction |
| `run_hdbscan` | no | no | standalone clustering |
| `compute_centroids` | no | no | centroid calculation |
| `match_topics` | no | no | topic continuity logic |
| `get_label` | no | yes (on cache miss) | labeling individual clusters |
| `append_trends` | no | no | writing time-series |
| `get_emerging_topics` | no | no | spike signal from existing TSV |
| `run` | yes | yes | full daily pipeline |

**Prerequisites:**
- Sections 1–7 run with **no API key** (synthetic data throughout).
- Section 8 (LLM labeling demo) requires `OPENAI_API_KEY`.
- Section 9 (full pipeline) requires `OPENAI_API_KEY` + FAISS vectorstore at `data/vectorstore/feeds/`.
- Section 10 is the error handling reference — no key needed.

## Sections
1. [Setup](#1-setup)
2. [Sample data — synthetic vectors](#2-sample-data--synthetic-vectors)
3. [Core clustering](#3-core-clustering)
4. [Topic continuity](#4-topic-continuity)
5. [Centroid and label cache I/O](#5-centroid-and-label-cache-io)
6. [Time-series output](#6-time-series-output)
7. [Signal generation](#7-signal-generation)
8. [LLM labeling (API-key gated)](#8-llm-labeling)
9. [Full pipeline (API-key + FAISS gated)](#9-full-pipeline)
10. [Error handling reference](#10-error-handling-reference)

---
## 1 — Setup

Import project modules and configure logging. The `sys.path` insertion ensures imports work when running from the `notebooks/` subdirectory.

In [ ]:
import json
import logging
import os
import sys
import tempfile
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

# Ensure the project root is on sys.path when running from notebooks/
PROJECT_ROOT = Path("..") if Path("../cluster_topics.py").exists() else Path(".")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from cluster_topics import (
    ClusteringAborted,
    DuplicateDateError,
    append_trends,
    compute_centroids,
    compute_spike,
    get_emerging_topics,
    get_label,
    load_centroids,
    load_label_cache,
    match_topics,
    reduce_dimensions,
    run_hdbscan,
    save_centroids,
    save_label_cache,
)
from constants import (
    CLUSTER_MAX_NOISE_RATIO,
    CLUSTER_MIN_CLUSTERS,
    CLUSTER_MIN_SAMPLES,
    CLUSTER_MIN_SIZE,
    CLUSTER_WINDOW_DAYS,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    force=True,
)

print(f"CLUSTER_WINDOW_DAYS     : {CLUSTER_WINDOW_DAYS}")
print(f"CLUSTER_MIN_SIZE        : {CLUSTER_MIN_SIZE}")
print(f"CLUSTER_MIN_SAMPLES     : {CLUSTER_MIN_SAMPLES}")
print(f"CLUSTER_MAX_NOISE_RATIO : {CLUSTER_MAX_NOISE_RATIO}")
print(f"CLUSTER_MIN_CLUSTERS    : {CLUSTER_MIN_CLUSTERS}")
RUN_DATE = date(2026, 3, 23)  # target date used throughout the notebook


---
## 2 — Sample data — synthetic vectors

All sections 3–7 use this synthetic corpus instead of the live FAISS vectorstore. This means no API key or local index is required to follow the guide.

The corpus mimics the real-world distribution observed on the CNBC feed:
- **1 800 articles** — matches a typical 45-day rolling window (~1 717 observed)
- **5 tight clusters** of 200 articles each (the "narratives"), embedded around distinct centers
- **800 noise points** — uniformly distributed, simulating the ~60% noise HDBSCAN naturally assigns to one-off news stories
- **1 536 dimensions** — matches `text-embedding-3-small`

The metadata DataFrame mirrors `feeds_registry.tsv` exactly: `id`, `date`, `title`, `link`, `guid`.

In [ ]:
rng = np.random.default_rng(seed=0)

DIM           = 1536
N_CLUSTERS    = 5
CLUSTER_SIZE  = 200
N_NOISE       = 800
N_TOTAL       = N_CLUSTERS * CLUSTER_SIZE + N_NOISE

CLUSTER_TOPICS = [
    "Iran oil geopolitics",
    "Fed rate pause bets",
    "AI chip supply constraints",
    "US-China tariff escalation",
    "Healthcare pharma earnings",
]

# Build tight cluster blobs
centers = rng.standard_normal((N_CLUSTERS, DIM)).astype(np.float32)
cluster_vecs = np.vstack([
    centers[i] + rng.standard_normal((CLUSTER_SIZE, DIM)).astype(np.float32) * 0.05
    for i in range(N_CLUSTERS)
])

# Build noise (uniform random — no cluster structure)
noise_vecs = rng.standard_normal((N_NOISE, DIM)).astype(np.float32)

VECTORS = np.vstack([cluster_vecs, noise_vecs])

# Ground-truth labels (for validation only — not passed to HDBSCAN)
TRUE_LABELS = np.array(
    [i for i in range(N_CLUSTERS) for _ in range(CLUSTER_SIZE)] + [-1] * N_NOISE
)

# Build matching metadata DataFrame
REF_DATE = date(2026, 3, 13)
dates = [REF_DATE - timedelta(days=i % 45) for i in range(N_TOTAL)]
META = pd.DataFrame({
    "id":    list(range(N_TOTAL)),
    "date":  pd.to_datetime(dates),
    "title": [
        CLUSTER_TOPICS[TRUE_LABELS[i]] if TRUE_LABELS[i] >= 0 else f"One-off article {i}"
        for i in range(N_TOTAL)
    ],
    "link":  [f"https://cnbc.com/{i}" for i in range(N_TOTAL)],
    "guid":  [f"guid-{i}" for i in range(N_TOTAL)],
})

print(f"VECTORS shape : {VECTORS.shape}")
print(f"META shape    : {META.shape}")
print(f"Date range    : {META['date'].min().date()} -> {META['date'].max().date()}")
print(f"Cluster articles : {N_CLUSTERS * CLUSTER_SIZE}  ({N_CLUSTERS} x {CLUSTER_SIZE})")
print(f"Noise articles   : {N_NOISE}  ({N_NOISE / N_TOTAL:.0%} of total)")

---
## 3 — Core clustering

The two core steps between raw vectors and cluster labels.

### `reduce_dimensions(vectors, n_components=50)`

Wraps `sklearn.decomposition.PCA` with a fixed `random_state=42` for determinism. Always fit on the window corpus, not all-time data — this keeps the PCA geometry relevant to the current 45-day narrative landscape.

**Why PCA before HDBSCAN?** HDBSCAN degrades in high dimensions (the "curse of dimensionality"). Reducing from 1 536 → 50 dimensions removes noise axes while preserving the dominant semantic structure. PCA explains ~52% of variance at 50 components on the real corpus.

### `run_hdbscan(X, min_cluster_size, min_samples, max_noise_ratio, min_clusters)`

| Parameter | Default | Effect |
|---|---|---|
| `min_cluster_size` | 10 | Minimum articles to form a cluster. Raise to get fewer, larger clusters. |
| `min_samples` | 3 | Controls how conservative cluster boundaries are. |
| `max_noise_ratio` | 0.90 | Abort threshold. 60–70% noise is **normal** for news data. |
| `min_clusters` | 3 | Abort if fewer than this many clusters form. |

Returns `(labels, noise_ratio)`. Label `-1` = noise (article doesn't belong to any recurring narrative).

In [ ]:
# Step 1: reduce 1536-dim → 50-dim
X = reduce_dimensions(VECTORS, n_components=50)
print(f"Input shape  : {VECTORS.shape}")
print(f"Reduced shape: {X.shape}")

# Step 2: cluster
labels, noise_ratio = run_hdbscan(X, min_cluster_size=10, min_samples=3)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
sizes = pd.Series(labels[labels != -1]).value_counts().sort_values(ascending=False)

print(f"\nClusters found : {n_clusters}")
print(f"Noise ratio    : {noise_ratio:.1%}  ({(labels == -1).sum()} / {len(labels)} articles)")
print(f"\nCluster sizes (top 10):")
for cid, sz in sizes.head(10).items():
    print(f"  cluster {int(cid):>2} : {sz} articles")

In [ ]:
# Verify clusters recovered the ground-truth narratives.
# Each HDBSCAN cluster should map predominantly to one true narrative.
print("Cluster -> dominant narrative (purity check):")
for cid in sorted(set(labels)):
    if cid == -1:
        continue
    member_true = TRUE_LABELS[labels == cid]
    dominant = pd.Series(member_true[member_true != -1]).mode()
    dominant_name = CLUSTER_TOPICS[dominant[0]] if len(dominant) else "noise"
    purity = (member_true == dominant[0]).mean() if len(dominant) else 0
    print(f"  cluster {int(cid):>2} : {dominant_name:<35}  purity={purity:.0%}")

---
## 4 — Topic continuity

The cluster IDs that HDBSCAN assigns (0, 1, 2 …) are **arbitrary and change every run**. Without continuity tracking, the time-series is broken — "cluster 3 today" is not the same as "cluster 3 tomorrow".

The solution: represent each cluster by its **centroid** (mean vector), then match today's centroids to yesterday's via cosine similarity. Matches above `threshold=0.85` inherit the prior `topic_id`; unmatched clusters receive a fresh UUID.

### `compute_centroids(vectors, labels)`

Mean of all member article embeddings per cluster. Noise (`-1`) is never a key.

### `match_topics(new_centroids, prior_topics, threshold=0.85, run_date)`

Greedy cosine-similarity assignment:
1. Build an (n_new × n_prior) similarity matrix.
2. Sort all pairs by similarity descending.
3. Assign greedily — each prior topic matches at most one new cluster.
4. Pairs below `threshold` get a fresh UUID regardless.

Returns `dict[cluster_int_id → centroid_entry]` where each entry has: `topic_id`, `label`, `centroid` (list), `first_seen`, `last_seen`.

In [ ]:
# Compute centroids from today's clustering result
centroids = compute_centroids(VECTORS, labels)

print(f"Clusters with centroids: {len(centroids)}")
for cid, vec in sorted(centroids.items()):
    print(f"  cluster {int(cid):>2} : centroid shape={vec.shape}, norm={np.linalg.norm(vec):.4f}")

In [ ]:
import uuid as _uuid

# Simulate a prior run: two of today's clusters existed before
prior_topics = {}
for cid in list(centroids.keys())[:2]:
    tid = str(_uuid.uuid4())
    vec = centroids[cid].copy()
    # Slightly perturb centroid to mimic natural drift
    vec += np.random.default_rng(99).standard_normal(vec.shape).astype(np.float32) * 0.02
    prior_topics[tid] = {
        "topic_id": tid,
        "label": f"Prior label {cid}",
        "centroid": vec.tolist(),
        "first_seen": "2026-03-01",
        "last_seen": "2026-03-22",
    }

from cluster_topics import match_topics

topic_map = match_topics(centroids, prior_topics, threshold=0.85, run_date=RUN_DATE)

matched = sum(1 for e in topic_map.values() if e["first_seen"] == "2026-03-01")
fresh   = len(topic_map) - matched

print(f"Clusters matched to prior topics : {matched}")
print(f"New topic IDs assigned           : {fresh}")
print()
for cid, entry in sorted(topic_map.items()):
    status = "MATCHED" if entry["first_seen"] == "2026-03-01" else "NEW    "
    print(f"  cluster {int(cid):>2} [{status}] topic_id={entry['topic_id'][:8]}..."          f"  first_seen={entry['first_seen']}")


---
## 5 — Centroid and label cache I/O

Two functions persist topic state across daily runs:

| Function | Reads / Writes | Safe on first run? |
|---|---|---|
| `load_centroids(path)` | JSON dict `{topic_id -> entry}` | Yes — returns `{}` if absent |
| `save_centroids(data, path)` | Atomic write via `.json.tmp` + `os.replace` | Yes |
| `load_label_cache(path)` | JSON dict `{topic_id -> label}` | Yes — returns `{}` if absent |
| `save_label_cache(cache, path)` | Atomic write via `.json.tmp` + `os.replace` | Yes |

**Invariant:** both files use `data/topic_centroids.json` and `data/topic_labels.json` by default (configured in `constants.py`). Pass an explicit `path` to redirect for testing.


In [ ]:
import tempfile, pathlib
from cluster_topics import load_centroids, save_centroids, load_label_cache, save_label_cache

with tempfile.TemporaryDirectory() as tmpdir:
    centroids_path = pathlib.Path(tmpdir) / "topic_centroids.json"
    labels_path    = pathlib.Path(tmpdir) / "topic_labels.json"

    # First run: files absent -> returns empty dicts
    assert load_centroids(centroids_path) == {}
    assert load_label_cache(labels_path) == {}
    print("First run (no files): both return {} as expected")

    # Build a centroid store from today's topic_map
    centroid_store = {entry["topic_id"]: entry for entry in topic_map.values()}
    save_centroids(centroid_store, centroids_path)
    print(f"Saved {len(centroid_store)} centroid entries")

    # Build a label cache
    label_cache = {entry["topic_id"]: f"Test label {i}" for i, entry in enumerate(topic_map.values())}
    save_label_cache(label_cache, labels_path)
    print(f"Saved {len(label_cache)} label entries")

    # Round-trip: reload and verify
    reloaded_centroids = load_centroids(centroids_path)
    reloaded_labels    = load_label_cache(labels_path)

    assert set(reloaded_centroids.keys()) == set(centroid_store.keys())
    assert set(reloaded_labels.keys()) == set(label_cache.keys())
    print("Round-trip verified: keys match after save/load")

    # Confirm atomic write: no .tmp file left behind
    tmp_files = list(pathlib.Path(tmpdir).glob("*.tmp"))
    assert tmp_files == [], f"Leftover .tmp files: {tmp_files}"
    print("Atomic write confirmed: no .tmp files after save")


---
## 6 — Time-series output (`append_trends`)

`append_trends` writes the daily count per topic to an append-only TSV:

```
data/topic_trends.tsv
  date         topic_id                              topic_label              article_count
  2026-03-22   3b4e...                               Fed rate pause bets      34
  2026-03-22   a7f1...                               Big tech earnings        21
  2026-03-23   3b4e...                               Fed rate pause bets      29
```

Key properties:
- **Append-only**: existing rows are never modified.
- **Atomic**: temp file + `os.replace` prevents partial writes.
- **Idempotent guard**: raises `DuplicateDateError` if the same date is submitted twice.
- Header row is written automatically on first run.


In [ ]:
import tempfile, pathlib, pandas as pd
from cluster_topics import append_trends, DuplicateDateError

# Build synthetic trend rows from today's topic_map
trend_rows = [
    {
        "date": str(RUN_DATE),
        "topic_id": entry["topic_id"],
        "topic_label": f"Label {cid}",
        "article_count": int((labels == cid).sum()),
    }
    for cid, entry in topic_map.items()
]

with tempfile.TemporaryDirectory() as tmpdir:
    tsv_path = pathlib.Path(tmpdir) / "topic_trends.tsv"

    # First append: creates the file with header
    append_trends(RUN_DATE, trend_rows, tsv_path)
    df = pd.read_csv(tsv_path, sep="\t")
    print(f"After first append: {len(df)} rows")
    print(df.to_string(index=False))

    # Second append on a different date succeeds
    from datetime import date
    next_date = date(2026, 3, 24)
    next_rows = [{**r, "date": str(next_date)} for r in trend_rows]
    append_trends(next_date, next_rows, tsv_path)
    df2 = pd.read_csv(tsv_path, sep="\t")
    print(f"\nAfter second append (different date): {len(df2)} rows, {df2['date'].nunique()} unique dates")

    # Duplicate date raises DuplicateDateError
    try:
        append_trends(RUN_DATE, trend_rows, tsv_path)
        print("ERROR: should have raised DuplicateDateError")
    except DuplicateDateError as e:
        print(f"\nDuplicateDateError correctly raised: {e}")


---
## 7 — Signal generation

Two functions detect emerging narratives from the trends history:

### `compute_spike(topic_id, trends, target_date, lookback_days=7, min_history=3)`

Returns `today_count / mean(prior_N_days)` or `None` when there is insufficient history.

| `None` condition | Reason |
|---|---|
| Topic not in trends | No prior observations |
| `< min_history` prior days | Too little history to form a baseline |
| Rolling average is 0 | Avoids divide-by-zero |

### `get_emerging_topics(target_date, trends, min_article_count=5, spike_lookback=7)`

Filters to topics above `min_article_count`, computes spikes, and returns a list sorted by `spike_ratio` descending.  
Each result: `{topic_id, label, spike_ratio, article_count}`.


In [ ]:
import pandas as pd
from datetime import date, timedelta
from cluster_topics import compute_spike, get_emerging_topics

# Build a synthetic 10-day trends dataset with one spiking topic
SPIKE_ID = "topic-spike-001"
QUIET_ID = "topic-quiet-002"
NEW_ID   = "topic-new-003"

rows = []
base = date(2026, 3, 14)
for delta in range(9):
    d = base + timedelta(days=delta)
    rows.append({"date": str(d), "topic_id": SPIKE_ID, "topic_label": "AI chip demand",  "article_count": 5})
    rows.append({"date": str(d), "topic_id": QUIET_ID, "topic_label": "Retail earnings", "article_count": 3})

# On target day: SPIKE_ID surges; QUIET_ID unchanged; NEW_ID first appearance
target = date(2026, 3, 23)
rows.append({"date": str(target), "topic_id": SPIKE_ID, "topic_label": "AI chip demand",  "article_count": 40})
rows.append({"date": str(target), "topic_id": QUIET_ID, "topic_label": "Retail earnings", "article_count": 4})
rows.append({"date": str(target), "topic_id": NEW_ID,   "topic_label": "New topic",       "article_count": 12})

trends_df = pd.DataFrame(rows)

spike_ratio = compute_spike(SPIKE_ID, trends_df, target)
quiet_ratio = compute_spike(QUIET_ID, trends_df, target)
new_ratio   = compute_spike(NEW_ID,   trends_df, target)

print(f"SPIKE_ID  spike_ratio : {spike_ratio}   (expected ~8.0)")
print(f"QUIET_ID  spike_ratio : {quiet_ratio}    (expected ~1.33)")
print(f"NEW_ID    spike_ratio : {new_ratio}        (expected None -- no history)")

# get_emerging_topics: min_article_count=5 filters out QUIET_ID (4 articles) and NEW_ID (no history)
emerging = get_emerging_topics(target, trends_df, min_article_count=5)
print(f"\nEmerging topics (min_count=5): {len(emerging)}")
for t in emerging:
    print(f"  {t['label']:<25}  spike={t['spike_ratio']:.2f}  count={t['article_count']}")


---
## 8 — LLM labeling (`get_label`)

`get_label` assigns a short human-readable label to a topic using `gpt-4o-mini`.
It is **cache-first**: once a label is stored in `topic_labels.json` for a given `topic_id`, the LLM is never called again for that topic.

```
0-3 LLM API calls per day in steady state
(only new topic_ids that have no cached label)
```

**Requires `OPENAI_API_KEY`** for the real path. You can pass a custom `llm_fn` to replace the real LLM with any callable `(articles: list[str]) -> str` — used in tests to stay offline.


In [ ]:
import os
from cluster_topics import get_label

# --- Offline demo: custom llm_fn (no API key needed) ---
def mock_llm(articles):
    return f"Mock label ({len(articles)} articles)"

cache = {}
tid   = "fake-topic-id-001"
articles = ["Nvidia surges on AI demand", "AMD eyes data center share", "Intel restructures"]

# Cache miss: calls mock_llm and stores result
label1 = get_label(tid, cache, articles, llm_fn=mock_llm)
print(f"Cache miss  -> label : {label1!r}")
print(f"Cache after first call: {cache}")

# Cache hit: returns immediately without calling llm_fn
def explode(_):
    raise RuntimeError("LLM called on cache hit -- bug!")

label2 = get_label(tid, cache, articles, llm_fn=explode)
print(f"Cache hit   -> label : {label2!r}  (no LLM call)")

# --- Online demo: real gpt-4o-mini (requires OPENAI_API_KEY) ---
if not os.environ.get("OPENAI_API_KEY"):
    print("\nOPENAI_API_KEY not set -- skipping real LLM call.")
else:
    real_cache = {}
    real_tid   = "real-topic-id-001"
    real_articles = [
        "Fed holds rates steady", "Treasury yields fall", "Powell signals caution",
        "Rate cut hopes grow", "Bond market rallies",
    ]
    real_label = get_label(real_tid, real_cache, real_articles)
    print(f"\nLabel from gpt-4o-mini : {real_label!r}")
    print(f"Cache updated          : {real_cache}")


---
## 9 — Full pipeline (`run`)

`run()` executes all 7 steps in sequence for a single day.

```python
summary = run(
    target_date    = date.today(),   # defaults to today
    window_days    = 45,             # rolling window
    centroids_path = Path("data/topic_centroids.json"),
    labels_path    = Path("data/topic_labels.json"),
    trends_path    = Path("data/topic_trends.tsv"),
    clusters_dir   = Path("data/topic_clusters"),
    skip_labeling  = False,
)
```

**Returns:**
```
{
  "date":            "2026-03-23",
  "window_articles": 1717,
  "n_clusters":      19,
  "noise_ratio":     0.598,
  "new_labels":      3,
  "matched_topics":  16,
  "cluster_sizes":   {"uuid-...": 42, ...}
}
```

**Requires:**
1. `OPENAI_API_KEY` in environment
2. FAISS vectorstore at `data/vectorstore/feeds/`

**Exit code 2** from the CLI signals `ClusteringAborted` — CI treats this as a warning, not a failure.


In [ ]:
import os, pathlib, tempfile
from datetime import date

_has_api_key = bool(os.environ.get("OPENAI_API_KEY"))
_has_faiss   = pathlib.Path("data/vectorstore/feeds/index.faiss").exists()

if not (_has_api_key and _has_faiss):
    missing = []
    if not _has_api_key: missing.append("OPENAI_API_KEY")
    if not _has_faiss:   missing.append("data/vectorstore/feeds/")
    print(f"Skipping run() demo -- missing: {', '.join(missing)}")
    print("To run: set OPENAI_API_KEY and ensure the FAISS store is built (python embed_feeds.py).")
else:
    import pprint
    from cluster_topics import run, ClusteringAborted

    with tempfile.TemporaryDirectory() as tmpdir:
        td = pathlib.Path(tmpdir)
        try:
            summary = run(
                target_date    = date(2026, 3, 23),
                centroids_path = td / "topic_centroids.json",
                labels_path    = td / "topic_labels.json",
                trends_path    = td / "topic_trends.tsv",
                clusters_dir   = td / "topic_clusters",
                skip_labeling  = False,
            )
            print("run() completed successfully:")
            pprint.pprint(summary)
        except ClusteringAborted as e:
            print(f"ClusteringAborted (degenerate run): {e}")


---
## 10 — Error handling reference

| Function | Raises | When |
|---|---|---|
| `run_hdbscan` | `ClusteringAborted` | noise ratio > `max_noise_ratio` OR n_clusters < `min_clusters` |
| `append_trends` | `DuplicateDateError` | `run_date` already present in the TSV |
| `extract_window_vectors` | `AssertionError` | row count mismatch vectors vs metadata (invariant violation) |
| `match_topics` | *(none)* | always returns a result (worst case: all new UUIDs) |
| `get_label` | *(none)* | LLM failure returns `"Unknown topic"` instead of raising |
| `load_centroids` | *(none)* | returns `{}` when file absent |
| `load_label_cache` | *(none)* | returns `{}` when file absent |

The two exceptions you will encounter in practice:
- **`ClusteringAborted`**: degenerate run. CI treats exit code 2 as a warning — log and skip.
- **`DuplicateDateError`**: same-date re-run. Manually delete the duplicate rows from `topic_trends.tsv` before re-running.


In [ ]:
import tempfile, pathlib, numpy as np
from datetime import date
from cluster_topics import (
    run_hdbscan, ClusteringAborted,
    append_trends, DuplicateDateError,
)

# --- ClusteringAborted: all-noise data ---
rng3 = np.random.default_rng(seed=42)
noise_data = rng3.standard_normal((300, 50)).astype(np.float32) * 5.0
try:
    run_hdbscan(noise_data, min_cluster_size=50, min_samples=10,
                max_noise_ratio=0.50, min_clusters=3)
    print("ERROR: should have raised ClusteringAborted")
except ClusteringAborted as e:
    print(f"ClusteringAborted (high noise): {e}")

# --- ClusteringAborted: too few clusters ---
centers2 = rng3.standard_normal((3, 50)).astype(np.float32) * 10
blob_data = np.vstack([
    centers2[i] + rng3.standard_normal((50, 50)).astype(np.float32) * 0.05
    for i in range(3)
])
try:
    run_hdbscan(blob_data, min_cluster_size=5, min_samples=2,
                max_noise_ratio=0.95, min_clusters=10)
    print("ERROR: should have raised ClusteringAborted")
except ClusteringAborted as e:
    print(f"ClusteringAborted (too few clusters): {e}")

# --- DuplicateDateError: same-date re-run ---
with tempfile.TemporaryDirectory() as tmpdir:
    tsv_path = pathlib.Path(tmpdir) / "topic_trends.tsv"
    rows = [{"date": "2026-03-23", "topic_id": "tid-001",
             "topic_label": "Test", "article_count": 10}]
    run_date = date(2026, 3, 23)
    append_trends(run_date, rows, tsv_path)   # first write succeeds
    try:
        append_trends(run_date, rows, tsv_path)  # second write same date raises
        print("ERROR: should have raised DuplicateDateError")
    except DuplicateDateError as e:
        print(f"DuplicateDateError: {e}")
